This notebook is used to split the data into multiple datasets:
1. A train&test dataset for R&R
2. Daily datasets used to simulate production daily data

The train&test dataset is stored locally on the block volume, while the daily datasets are stored in OCI Object Storage under different object names.

In [2]:
import io
import os
import pandas as pd
import oci

In [4]:
signer = oci.auth.signers.get_resource_principals_signer()
object_storage_client = oci.object_storage.ObjectStorageClient(
    config={},
    signer=signer
)
namespace_name = object_storage_client.get_namespace().data
bucket_name = 'anomaly-detection-regression'

In [16]:
path='/home/datascience/anomaly_detection_project/regression/data/walmart_cleaned.csv'
df = pd.read_csv(path)

date_col = 'Date'
df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col)
unique_dates = df[date_col].drop_duplicates().sort_values()

In [17]:
df[date_col] = pd.to_datetime(df[date_col])
train_dates = unique_dates[:130]
future_dates = unique_dates[130:]
train_df = df[df[date_col].isin(train_dates)].copy()
train_df.shape

(383040, 17)

In [19]:
train_df.to_csv('/home/datascience/anomaly_detection_project/regression/data/walmart_train.csv')

In [20]:
for current_date in future_dates:

    daily_df = df[df[date_col] == current_date].copy()

    date_str = current_date.strftime('%Y-%m-%d')

    object_name = f'daily_files/{date_str}.csv'

    csv_bytes = daily_df.to_csv(index=False).encode('utf-8')

    object_storage_client.put_object(
        namespace_name=namespace_name,
        bucket_name=bucket_name,
        object_name=object_name,
        put_object_body=csv_bytes,
        content_type='text/csv'
    )

    print(f'Uploaded: {object_name}')

Uploaded: daily_files/2012-08-03.csv
Uploaded: daily_files/2012-08-10.csv
Uploaded: daily_files/2012-08-17.csv
Uploaded: daily_files/2012-08-24.csv
Uploaded: daily_files/2012-08-31.csv
Uploaded: daily_files/2012-09-07.csv
Uploaded: daily_files/2012-09-14.csv
Uploaded: daily_files/2012-09-21.csv
Uploaded: daily_files/2012-09-28.csv
Uploaded: daily_files/2012-10-05.csv
Uploaded: daily_files/2012-10-12.csv
Uploaded: daily_files/2012-10-19.csv
Uploaded: daily_files/2012-10-26.csv


In [ ]:
for current_date in future_dates:

    daily_df = df[df[date_col] == current_date].copy()

    date_str = current_date.strftime('%Y-%m-%d')

    object_name = f'daily_files/{date_str}.csv'

    csv_bytes = daily_df.to_csv(index=False).encode('utf-8')

    object_storage_client.put_object(
        namespace_name=namespace_name,
        bucket_name=bucket_name,
        object_name=object_name,
        put_object_body=csv_bytes,
        content_type='text/csv'
    )

    print(f'Uploaded: {object_name}')